In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import zipfile
import io

In [13]:
from requests import Response
print("\n--- FASE 1: Extraccion ---")

url = "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/08/public_data_web_2026-07.csv"

csv_file_name = "2026-07.csv"
print(f"Descargando datos de: {url}")
try:
    response = requests.get(url, timeout=1200)
    response.raise_for_status()
    print("Descarga completa con exito.")

except requests.exceptions.Timeout as e:
    print(f"Error durante la descarga: {e}")
    df_raw = pd.DataFrame()
except requests.exceptions.RequestException as e:
    print(f"Error durante la descarga: {e}")
    df_raw = pd.DataFrame()


--- FASE 1: Extraccion ---
Descargando datos de: https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/08/public_data_web_2026-07.csv
Descarga completa con exito.


In [14]:
with open(csv_file_name, 'wb') as f:
    f.write(response.content)
print(f"Archivo CSV guardado como {csv_file_name}")

print(f"Leyendo datos desde: {csv_file_name}")
df_raw = pd.read_csv(csv_file_name)
print("Extraccion completada.")
print(f"Se cargaron {df_raw.shape[0]} registros")


Archivo CSV guardado como 2026-07.csv
Leyendo datos desde: 2026-07.csv
Extraccion completada.
Se cargaron 1493484 registros


In [19]:
print("Tamano del DataFrame")
print(df_raw.shape)

print("\nPrevisualizacion del DataFrame")
print(df_raw.head(10))

Tamano del DataFrame
(1493484, 9)

Previsualizacion del DataFrame
  Genero_Usuario  Edad_Usuario     Bici Ciclo_Estacion_Retiro Fecha_Retiro  \
0              F          26.0  5552989                   085   30/06/2026   
1              M          33.0  5128335                   259   30/06/2026   
2              M          34.0  8647703                   040   30/06/2026   
3              M          34.0  5633250                   492   30/06/2026   
4              O          41.0  8516015                   133   30/06/2026   
5              M          35.0  6215123                   013   30/06/2026   
6              M          45.0  3861267                   465   30/06/2026   
7              M          45.0  6013460                   449   30/06/2026   
8              M          41.0  5971320                   011   30/06/2026   
9              M          27.0  8516676                   451   30/06/2026   

  Hora_Retiro Ciclo_EstacionArribo Fecha_Arribo Hora_Arribo  
0    23:43:41

In [23]:
# --- 2.2 Feature Engineering ---
print("\nIniciando Feature Engineering...")

# Para calcular la duración real, necesitamos combinar la fecha con la hora específica
df_raw['Fecha_Retiro_Completa'] = pd.to_datetime(df_raw['Fecha_Retiro'].dt.date.astype(str) + ' ' + df_raw['Hora_Retiro'])
df_raw['Fecha_Arribo_Completa'] = pd.to_datetime(df_raw['Fecha_Arribo'].dt.date.astype(str) + ' ' + df_raw['Hora_Arribo'])

# 1. Duración del viaje en minutos (usando las columnas completas)
df_raw['duracion_minutos'] = (df_raw['Fecha_Arribo_Completa'] - df_raw['Fecha_Retiro_Completa']).dt.total_seconds() / 60

# 2. Día de la semana (0=Lunes, 6=Domingo)
df_raw['dia_semana'] = df_raw['Fecha_Retiro'].dt.dayofweek

# 3. Hora del día
df_raw['hora_inicio'] = df_raw['Fecha_Retiro_Completa'].dt.hour

# 4. Categoría de día (Fin de semana vs. Entre semana)
df_raw['tipo_dia'] = df_raw['dia_semana'].apply(lambda x: 'Fin de Semana' if x >= 5 else 'Entre Semana')

print("Nuevas características creadas con precisión: 'duracion_minutos', 'dia_semana', 'hora_inicio', 'tipo_dia'.")
display(df_raw[['Fecha_Retiro_Completa', 'Fecha_Arribo_Completa', 'duracion_minutos']].head())


Iniciando Feature Engineering...


AttributeError: Can only use .dt accessor with datetimelike values

In [ ]:
# --- 2.3 Normalización / Estandarización ---
# Vamos a normalizar la duración del viaje para que esté en una escala de 0 a 1.
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_raw['duracion_normalizada'] = scaler.fit_transform(df_raw[['duracion_minutos']])
print("\n'duracion_minutos' normalizada a una escala de 0 a 1.")

# --- 2.4 Encoding de Variables Categóricas ---
# La columna 'tipo_dia' es categórica. La convertiremos a números usando One-Hot Encoding.
df_encoded = pd.get_dummies(df_raw, columns=['tipo_dia'], prefix='dia')
print("Variable 'tipo_dia' codificada con One-Hot Encoding.")

# --- 2.5 Balanceo de Clases ---
# Imaginemos que queremos predecir si un viaje es "muy largo" (> 60 min).
df_raw['viaje_largo'] = df_raw['duracion_minutos'] > 60
print("\nEjemplo de Balanceo de Clases:")
print("Distribución de 'viaje_largo' antes del balanceo:")
print(df_raw['viaje_largo'].value_counts())

# --- Verificación del DataFrame Transformado ---
print("\n--- Vista previa del DataFrame transformado ---")
print(df_encoded.head())